# Web Scraping & Analysis — quotes.toscrape.com

**Website:** [quotes.toscrape.com](https://quotes.toscrape.com/)  
**Tool:** BeautifulSoup + Requests  
**Goal:** Scrape all quotes with author metadata, save as CSV, and perform descriptive statistics with visualizations.

---
## Step 1 — Import Libraries

- **requests** — HTTP requests to download pages.
- **BeautifulSoup** — parse HTML and extract data.
- **pandas** — tabular data, statistics, CSV.
- **matplotlib / seaborn** — visualizations.
- **datetime** — parse author birth dates into numeric year.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('All libraries imported successfully!')

---
## Step 2 — Web Scraping from quotes.toscrape.com

### How the scraper works

1. We iterate through **all pages** of the site (`/page/1/`, `/page/2/`, …) until there is no "Next" button.
2. From each quote block (`<div class="quote">`) we extract:
   - **Quote text** — from `<span class="text">`.
   - **Author name** — from `<small class="author">`.
   - **Tags** — from `<a class="tag">`.
3. We also visit each **author's "about" page** (once per unique author) to get:
   - **Birth date** — from `<span class="author-born-date">`.
   - **Birth place** — from `<span class="author-born-location">`.
   - **Description** — from `<div class="author-description">`.
4. We derive additional **numerical features**:
   - `quote_length` — number of characters in the quote.
   - `word_count` — number of words.
   - `num_tags` — number of tags per quote.
   - `author_birth_year` — extracted from the birth date string.

In [ ]:
BASE = 'https://quotes.toscrape.com'
session = requests.Session()


def get_soup(url: str) -> BeautifulSoup:
    resp = session.get(url, timeout=15)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, 'lxml')


# ── Phase 1: scrape all quote pages ───────────────────────────────────
print('Phase 1 — Scraping quote pages …')
raw_quotes = []
author_links = {}  # author_name -> relative about-URL
page = 1

while True:
    soup = get_soup(f'{BASE}/page/{page}/')
    quotes_on_page = soup.select('div.quote')

    if not quotes_on_page:
        break

    for q in quotes_on_page:
        text = q.select_one('span.text').get_text(strip=True)
        # Remove surrounding curly quotes
        text = text.strip('\u201c\u201d')
        author = q.select_one('small.author').get_text(strip=True)
        tags = [tag.get_text(strip=True) for tag in q.select('a.tag')]

        about_link = q.select_one('a[href*="/author/"]')
        if about_link and author not in author_links:
            author_links[author] = about_link['href']

        raw_quotes.append({
            'quote_text': text,
            'author': author,
            'tags': ', '.join(tags),
            'num_tags': len(tags),
            'quote_length': len(text),
            'word_count': len(text.split()),
        })

    print(f'  Page {page} — {len(quotes_on_page)} quotes  (total: {len(raw_quotes)})')

    next_btn = soup.select_one('li.next a')
    if not next_btn:
        break
    page += 1

print(f'  Quotes done — {len(raw_quotes)} quotes from {len(author_links)} authors.\n')

# ── Phase 2: scrape author "about" pages ───────────────────────────────
print('Phase 2 — Scraping author pages …')
author_info = {}

for name, href in author_links.items():
    soup = get_soup(BASE + href)

    born_date_str = soup.select_one('span.author-born-date')
    born_date_str = born_date_str.get_text(strip=True) if born_date_str else None

    born_location = soup.select_one('span.author-born-location')
    born_location = born_location.get_text(strip=True).lstrip('in ') if born_location else None

    description = soup.select_one('div.author-description')
    description = description.get_text(strip=True) if description else None

    # Parse birth year from date string like "June 14, 1928"
    birth_year = None
    if born_date_str:
        try:
            birth_year = datetime.strptime(born_date_str, '%B %d, %Y').year
        except ValueError:
            pass

    author_info[name] = {
        'author_birth_date': born_date_str,
        'author_birth_year': birth_year,
        'author_birth_place': born_location,
        'author_description': description,
    }

print(f'  Fetched info for {len(author_info)} authors.\n')

# ── Phase 3: merge quotes + author info ───────────────────────────────
quotes = []
for q in raw_quotes:
    info = author_info.get(q['author'], {})
    quotes.append({**q, **info})

print(f'Done!  Total rows: {len(quotes)}')

---
## Step 3 — Save the Dataset to CSV

We convert the list of dictionaries into a pandas DataFrame and save it as `quotes_dataset.csv`.

In [ ]:
df = pd.DataFrame(quotes)
df.to_csv('quotes_dataset.csv', index=False)
print(f'Dataset saved to quotes_dataset.csv  ({df.shape[0]} rows, {df.shape[1]} columns)')
df.head(10)

---
## Step 4 — Load & Explore the Dataset

Reload from CSV and inspect structure: shape, data types, non-null counts, first rows.

In [ ]:
df = pd.read_csv('quotes_dataset.csv')
print(f'Shape: {df.shape}\n')
print('--- Data Types ---')
print(df.dtypes)
print('\n--- Info ---')
df.info()
print('\n--- First 5 Rows ---')
df.head()

---
## Step 5 — Descriptive Statistics

Numerical columns: `quote_length`, `word_count`, `num_tags`, `author_birth_year`.  
We compute: mean, median, mode, standard deviation, min, max, quartiles.

In [ ]:
num_cols = ['quote_length', 'word_count', 'num_tags', 'author_birth_year']

print('=== describe() ===')
print(df[num_cols].describe())

print('\n=== Mode ===')
for col in num_cols:
    mode_val = df[col].mode()[0]
    print(f'  {col}: {mode_val}')

print('\n=== Custom Summary Table ===')
summary = pd.DataFrame({
    'mean': df[num_cols].mean(),
    'median': df[num_cols].median(),
    'mode': df[num_cols].mode().iloc[0],
    'std': df[num_cols].std(),
    'min': df[num_cols].min(),
    'max': df[num_cols].max(),
})
summary

---
## Step 6 — Identify and Handle Missing Values

- **Deletion**: drop rows missing the quote text or author.
- **Imputation**: fill missing birth year with the median; fill missing text fields with placeholders.

In [ ]:
print('=== Missing Values Before Handling ===')
missing_before = df.isnull().sum()
print(missing_before[missing_before > 0] if missing_before.sum() > 0 else 'No missing values found!')
print(f'Total missing cells: {missing_before.sum()}')

# Deletion: drop rows where quote_text or author is missing
rows_before = len(df)
df.dropna(subset=['quote_text', 'author'], inplace=True)
print(f'\nRows dropped (missing quote_text/author): {rows_before - len(df)}')

# Imputation
df['author_birth_place'] = df['author_birth_place'].fillna('Unknown')
df['author_description'] = df['author_description'].fillna('No description available')
df['author_birth_date'] = df['author_birth_date'].fillna('Unknown')

if df['author_birth_year'].isnull().any():
    median_year = df['author_birth_year'].median()
    df['author_birth_year'] = df['author_birth_year'].fillna(median_year)
    print(f'  Imputed author_birth_year NaNs with median = {median_year}')

print('\n=== Missing Values After Handling ===')
missing_after = df.isnull().sum()
print(missing_after[missing_after > 0] if missing_after.sum() > 0 else 'No missing values remain!')
print(f'Final dataset shape: {df.shape}')

---
## Step 7 — Visualizations

1. **Box Plot** — distribution of quote length (numerical).
2. **Bar Chart** — number of quotes per author (categorical).
3. **Scatter Plot** — quote length vs. word count (two numerical variables).

### 7.1 — Box Plot: Distribution of Quote Length

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='quote_length', color='#5B9BD5', width=0.4, ax=ax)
ax.set_title('Distribution of Quote Length (characters)', fontsize=16, fontweight='bold')
ax.set_xlabel('Quote Length', fontsize=13)
plt.tight_layout()
plt.savefig('boxplot_quote_length.png', dpi=150)
plt.show()

### 7.2 — Bar Chart: Number of Quotes per Author

In [ ]:
author_counts = df['author'].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=author_counts.values, y=author_counts.index, palette='viridis', ax=ax)
ax.set_title('Number of Quotes per Author', fontsize=16, fontweight='bold')
ax.set_xlabel('Count', fontsize=13)
ax.set_ylabel('Author', fontsize=13)
plt.tight_layout()
plt.savefig('barchart_authors.png', dpi=150)
plt.show()

### 7.3 — Scatter Plot: Quote Length vs. Word Count

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    df['word_count'],
    df['quote_length'],
    alpha=0.6,
    edgecolors='white',
    linewidth=0.5,
    s=60,
    c=df['num_tags'],
    cmap='plasma'
)
cbar = plt.colorbar(ax.collections[0], ax=ax)
cbar.set_label('Number of Tags', fontsize=12)
ax.set_title('Quote Length vs. Word Count', fontsize=16, fontweight='bold')
ax.set_xlabel('Word Count', fontsize=13)
ax.set_ylabel('Quote Length (characters)', fontsize=13)
plt.tight_layout()
plt.savefig('scatter_length_words.png', dpi=150)
plt.show()

---
## Summary

| Step | What we did |
|------|-------------|
| 1 | Imported libraries |
| 2 | Scraped all quotes from quotes.toscrape.com — text, author, tags + author birth info |
| 3 | Saved the dataset to `quotes_dataset.csv` |
| 4 | Loaded CSV and explored structure |
| 5 | Descriptive statistics: mean, median, mode, std, min, max, quartiles |
| 6 | Identified and handled missing values (deletion + imputation) |
| 7 | Three visualizations: box plot (quote length), bar chart (authors), scatter plot (length vs. words) |